Import the data

In [1]:
import pandas as pd

df = pd.read_csv("data/restructured.csv")

In [2]:
df.head()

,roundup_title,topics,story_title,story_text,bias_label
0,"""Dream"" 50 Years Later",Civil Rights,Thousands Gather In D.C. To Mark 1963 Civil Ri...,People are assembling on the National Mall to ...,center
1,"""Dream"" 50 Years Later",Civil Rights,March In Washington To Continue Focus On Civil...,Alice Long planned months ago to use vacation ...,left
2,"""Dream"" 50 Years Later",Civil Rights,Remembering My Uncle's 'Dream',"Fifty years ago, a valiant group of people fro...",right
3,"""Good Shutdown"" in September?",Politics,"President Trump Calls for a ""Good Shutdown"" in...",President Donald Trump made a bold statement o...,right
4,"""Good Shutdown"" in September?",Politics,Trump: US ‘needs a good shutdown’,"President Trump on Tuesday called for a ""good ...",center


Preprocess the Data

In [3]:
df['full_text'] = df['story_title'] + " " + df['story_text']

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Download stopwords from NLTK
#nltk.download('punkt')
#nltk.download('stopwords')

def data_preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    
    new_tokens = []
    for w in tokens:
        if w not in stop_words and w.isalpha():  
            stem_tokens = ps.stem(w)
            new_tokens.append(stem_tokens)
    return new_tokens


# Apply preprocessing to each message
df['full_text'] = df['full_text'].apply(data_preprocess)

# Preview the data to ensure pre-processing is as expected
df.head(10)


,roundup_title,topics,story_title,story_text,bias_label,full_text
0,"""Dream"" 50 Years Later",Civil Rights,Thousands Gather In D.C. To Mark 1963 Civil Ri...,People are assembling on the National Mall to ...,center,"[thousand, gather, mark, civil, right, march, ..."
1,"""Dream"" 50 Years Later",Civil Rights,March In Washington To Continue Focus On Civil...,Alice Long planned months ago to use vacation ...,left,"[march, washington, continu, focu, civil, righ..."
2,"""Dream"" 50 Years Later",Civil Rights,Remembering My Uncle's 'Dream',"Fifty years ago, a valiant group of people fro...",right,"[rememb, uncl, fifti, year, ago, valiant, grou..."
3,"""Good Shutdown"" in September?",Politics,"President Trump Calls for a ""Good Shutdown"" in...",President Donald Trump made a bold statement o...,right,"[presid, trump, call, good, shutdown, septemb,..."
4,"""Good Shutdown"" in September?",Politics,Trump: US ‘needs a good shutdown’,"President Trump on Tuesday called for a ""good ...",center,"[trump, us, need, good, shutdown, presid, trum..."
5,"""Good Shutdown"" in September?",Politics,Trump's frustration with budget compromise has...,Congress looks set to enact a bipartisan spend...,left,"[trump, frustrat, budget, compromis, consid, m..."
6,"""Skinny Repeal"" Motions Fails",US Senate,Obamacare repeal is dead for now. What could t...,The Senate’s effort to repeal and replace Obam...,center,"[obamacar, repeal, dead, could, mean, senat, e..."
7,"""Skinny Repeal"" Motions Fails",US Senate,Why Senate Republicans couldn’t repeal Obamacare,"In a stunning turn, Senate Republicans — in th...",left,"[senat, republican, repeal, obamacar, stun, tu..."
8,"""Skinny Repeal"" Motions Fails",US Senate,Senate Fails To Pass Motion To Proceed On ‘Ski...,Senate Majority Leader Mitch McConnell failed ...,right,"[senat, fail, pass, motion, proceed, skinni, r..."
9,"""Trump University"" Documents Unsealed",Elections,Trump University ‘Playbooks’ Unsealed in Lawsu...,Trump University gave employees detailed instr...,right,"[trump, univers, playbook, unseal, lawsuit, se..."


In [5]:
df_multi = df.copy()

# DF for binary classification (only left and right)
df_binary = df[df['bias_label'].isin(['left', 'right'])].copy()
df_binary.reset_index(drop=True, inplace=True)


Bag of Words (for binary and multi-class classification)

In [6]:
# Convert DataFrame rows into a list of tuples, each containing message and label
ml_list_multi = [(m,l) for m,l in zip(df_multi['full_text'], df_multi['bias_label'])]
ml_list_binary = [(m,l) for m,l in zip(df_binary['full_text'], df_binary['bias_label'])]

# Create a list of all word tokens
w_list_multi = []
for w in df_multi['full_text']:
    w_list_multi.extend(w)

w_list_binary = []
for w in df_binary['full_text']:
    w_list_binary.extend(w)

# Create a frequency distribution of all words
w_count_multi = nltk.FreqDist(w_list_multi)
w_count_binary = nltk.FreqDist(w_list_binary)

# Use the top 1000 words as features
top_words_multi = list(w_count_multi.keys())[:1000]
top_words_binary = list(w_count_binary.keys())[:1000]

# Define a Feature Extraction Function
def extract_features(doc, top_words):
    #convert the doc into a set of unique words
    w_set = set(doc)
    # initializes an empty dictionary
    f_dict = {}
    for w in top_words:
        # checks whether that word is present in the words set. 
        # The result is a Boolean value (True if the word is present, False otherwise)
        f_dict[w] = True if w in w_set else False  
    return f_dict

# Created Feature Sets
f_sets_multi = [(extract_features(m, top_words_multi), l) for m, l in ml_list_multi]
f_sets_binary = [(extract_features(m, top_words_binary), l) for m, l in ml_list_binary]

# (3.2) Split 75% of the data for training, 25% for testing
size_multi = int(len(f_sets_multi)*0.75)
train_set_multi, test_set_multi = f_sets_multi[:size_multi], f_sets_multi[size_multi:]

size_binary = int(len(f_sets_binary)*0.75)
train_set_binary, test_set_binary = f_sets_binary[:size_binary], f_sets_binary[size_binary:]

In [7]:
print("Training set size multi-classification:", len(train_set_multi))
print("Test set size multi-classification:", len(test_set_multi))

print("Training set size binary-classification:", len(train_set_binary))
print("Test set size binary-classification:", len(test_set_binary))

Training set size multi-classification: 18378
Test set size multi-classification: 6127
Training set size binary-classification: 12603
Test set size binary-classification: 4202


Train Multi Class Naive Bayes Classifier (Bag of Words)

In [8]:
from nltk import NaiveBayesClassifier # For creating and using a Naive Bayes classifier
from collections import defaultdict # For dictionaries with default values for missing keys

# (4.1) Train the Naive Bayes Classifier
nb_multi = NaiveBayesClassifier.train(train_set_multi)

#(4.2) Most informative features
nb_multi.show_most_informative_features(20)

# (4.3) Get the list of actual class from the test set and predicted class from the classifier
predicted_label_multi=[]
actual_label_multi=[]
for f, l in test_set_multi:
    predicted_label_multi.append(nb_multi.classify(f)) 
    actual_label_multi.append(l)

Most Informative Features
                    bold = True             left : center =      7.6 : 1.0
                   huddl = True             left : center =      5.6 : 1.0
                  galvan = True             left : right  =      5.6 : 1.0
                 bernard = True            right : left   =      5.0 : 1.0
                 sequest = True             left : center =      4.5 : 1.0
                   crush = True             left : center =      4.4 : 1.0
                   tight = True           center : right  =      4.3 : 1.0
                    menu = True            right : center =      4.2 : 1.0
                  sprawl = True           center : right  =      3.7 : 1.0
                  silver = True            right : left   =      3.4 : 1.0
                  sierra = True            right : center =      3.4 : 1.0
                 regroup = True             left : center =      3.3 : 1.0
            dissatisfact = True           center : right  =      3.3 : 1.0

Train Binary Naive Bayes Classifier (Bag of Words)

In [9]:
# (4.1) Train the Naive Bayes Classifier
nb_binary = NaiveBayesClassifier.train(train_set_binary)

#(4.2) Most informative features
nb_binary.show_most_informative_features(20)

# (4.3) Get the list of actual class from the test set and predicted class from the classifier
predicted_label_binary=[]
actual_label_binary=[]
for f, l in test_set_binary:
    predicted_label_binary.append(nb_binary.classify(f)) 
    actual_label_binary.append(l)

Most Informative Features
                 bernard = True            right : left   =      5.0 : 1.0
                  unseat = True             left : right  =      5.0 : 1.0
                     dem = True            right : left   =      4.4 : 1.0
                  silver = True            right : left   =      3.4 : 1.0
                   huddl = True             left : right  =      3.4 : 1.0
                firework = True            right : left   =      3.2 : 1.0
                   naomi = True            right : left   =      3.0 : 1.0
              outperform = True            right : left   =      3.0 : 1.0
                  sensat = True            right : left   =      3.0 : 1.0
                 lifesav = True             left : right  =      3.0 : 1.0
                   unusu = True             left : right  =      2.9 : 1.0
                   drive = True             left : right  =      2.9 : 1.0
                precinct = True            right : left   =      2.8 : 1.0

Evaluation of Multi Class Naive Bayes Classifier with Bag of Words

In [10]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd

# Accuracy
accuracy = round(accuracy_score(actual_label_multi, predicted_label_multi), 2)
print(f"Accuracy: {accuracy * 100}%\n")

# Precision, recall, F1
print(classification_report(actual_label_multi, predicted_label_multi))

# Confusion matrix
cm = pd.DataFrame(confusion_matrix(actual_label_multi, predicted_label_multi),
                  index=sorted(set(actual_label_multi)),
                  columns=sorted(set(actual_label_multi)))
print("\nConfusion Matrix:")
print(cm)

Accuracy: 41.0%

              precision    recall  f1-score   support

      center       0.38      0.35      0.36      1941
        left       0.42      0.45      0.43      2098
       right       0.44      0.44      0.44      2088

    accuracy                           0.41      6127
   macro avg       0.41      0.41      0.41      6127
weighted avg       0.41      0.41      0.41      6127


Confusion Matrix:
        center  left  right
center     678   693    570
left       574   951    573
right      533   642    913


Evaluation of Binary Naive Bayes Classifier with Bag of Words

In [11]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd

# Accuracy
accuracy = round(accuracy_score(actual_label_binary, predicted_label_binary), 2)
print(f"Accuracy: {accuracy * 100}%\n")

# Precision, recall, F1
print(classification_report(actual_label_binary, predicted_label_binary))

# Confusion matrix
cm = pd.DataFrame(confusion_matrix(actual_label_binary, predicted_label_binary),
                  index=sorted(set(actual_label_binary)),
                  columns=sorted(set(actual_label_binary)))
print("\nConfusion Matrix:")
print(cm)

Accuracy: 60.0%

              precision    recall  f1-score   support

        left       0.60      0.60      0.60      2106
       right       0.60      0.59      0.59      2096

    accuracy                           0.60      4202
   macro avg       0.60      0.60      0.60      4202
weighted avg       0.60      0.60      0.60      4202


Confusion Matrix:
       left  right
left   1273    833
right   859   1237


TF-IDF multi-class classification 

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd

# Join tokens into strings
df_multi['text_str'] = df_multi['full_text'].apply(lambda x: ' '.join(x))

# TF-IDF vectorization
vectorizer_multi = TfidfVectorizer(max_features=1000)
X_multi = vectorizer_multi.fit_transform(df_multi['text_str'])
y_multi = df_multi['bias_label']

# Train/test split
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(X_multi, y_multi, test_size=0.25, random_state=42)

# Train Naive Bayes
model_multi = MultinomialNB()
model_multi.fit(X_train_multi, y_train_multi)

# Predict
y_pred_multi = model_multi.predict(X_test_multi)

# Evaluate
print("\n--- Multi-Class TF-IDF ---")
print(f"Accuracy: {accuracy_score(y_test_multi, y_pred_multi):.2%}")
print(classification_report(y_test_multi, y_pred_multi))
cm_multi = pd.DataFrame(confusion_matrix(y_test_multi, y_pred_multi),
                        index=model_multi.classes_,
                        columns=model_multi.classes_)
print("\nConfusion Matrix:\n", cm_multi)



--- Multi-Class TF-IDF ---
Accuracy: 40.53%
              precision    recall  f1-score   support

      center       0.38      0.20      0.26      1943
        left       0.39      0.56      0.46      2038
       right       0.44      0.45      0.44      2146

    accuracy                           0.41      6127
   macro avg       0.40      0.40      0.39      6127
weighted avg       0.40      0.41      0.39      6127


Confusion Matrix:
         center  left  right
center     381   934    628
left       288  1143    607
right      329   858    959


TF-IDF Binary Classification

In [16]:
# Join tokens into strings
df_binary['text_str'] = df_binary['full_text'].apply(lambda x: ' '.join(x))

# TF-IDF vectorization
vectorizer_binary = TfidfVectorizer(max_features=1000)
X_binary = vectorizer_binary.fit_transform(df_binary['text_str'])
y_binary = df_binary['bias_label']

# Train/test split
X_train_binary, X_test_binary, y_train_binary, y_test_binary = train_test_split(X_binary, y_binary, test_size=0.25, random_state=42)

# Train Naive Bayes
model_binary = MultinomialNB()
model_binary.fit(X_train_binary, y_train_binary)

# Predict
y_pred_binary = model_binary.predict(X_test_binary)

# Evaluate
print("\n--- Binary TF-IDF ---")
print(f"Accuracy: {accuracy_score(y_test_binary, y_pred_binary):.2%}")
print(classification_report(y_test_binary, y_pred_binary))
cm_binary = pd.DataFrame(confusion_matrix(y_test_binary, y_pred_binary),
                         index=model_binary.classes_,
                         columns=model_binary.classes_)
print("\nConfusion Matrix:\n", cm_binary)


--- Binary TF-IDF ---
Accuracy: 58.50%
              precision    recall  f1-score   support

        left       0.60      0.58      0.59      2146
       right       0.57      0.59      0.58      2056

    accuracy                           0.58      4202
   macro avg       0.59      0.59      0.58      4202
weighted avg       0.59      0.58      0.59      4202


Confusion Matrix:
        left  right
left   1244    902
right   842   1214
